___

# <font color= #99C8F5> **Tarea FFNN: NBA** </font>
#### <font color= #2E9AFE> `Modelos No Lineales Para Pronósticos`</font>
<Strong> Sofía Maldonado, Aissa Berenice </Strong>

_29/03/2026._

___

### <font color= #99C8F5> **Introducción** </font>


En esta práctica buscamos utilizar una FFNN para predecir los puntos que va a anotar un equipo de básquetbol en un partido específico. Elegimos los Golden State Warriors de la NBA y vamos a revisar los partidos que jugaron durante la temporada 2024-2025.

**NOTA SOBRE LA API**: Tuvimos muchísimos problemas para cargar datos con la API que vimos en la clase. La documentación es, a nuestro parecer, muy confusa y otros paquetes para conseguir datos de la NBA como [nba_api](https://github.com/swar/nba_api) tampoco funcionaron bien. Para tener datos con los cuales hacer la tarea, decidimos utilizar Basketball-Reference para conseguir los datos directamente. En [esta página](https://www.basketball-reference.com/teams/GSW/2025_games.html) se pueden ver datos generales de todos los 82 juegos de temporada regular que jugaron en 2024-2025, con información básica incluyendo los puntos que anotó el equipo. Los datos se pueden guardar en formato CSV, que es lo que hicimos para cargarlos en un DataFrame. 

Poder predecir los puntos que va a anotar un equipo puede ser útil para apuestas, pero dentro de la misma organización también puede ser conveniente para comparar las predicciones con los valores esperados de los equipos contra quienes vas a jugar después, con el objetivo de planear apropiadamente si el equipo va a ganar o perder un partido, y los pasos que se pudieran tomar para mitigar un resultado negativo.

In [29]:
# Imports
from nba_api.stats.endpoints.leaguedashteamstats import LeagueDashTeamStats
import pandas as pd
import numpy as np

from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

In [15]:
# Loading Data
df = pd.read_csv('warriors_data.csv')
df

,G,Date,Start (ET),Unnamed: 3,Unnamed: 4,Unnamed: 5,Opponent,Unnamed: 7,Unnamed: 8,Tm,Opp,W,L,Streak,Attend.,LOG,Notes
0,1,Wed Oct 23 2024,10:00p,NaN,Box Score,@,Portland Trail Blazers,W,NaN,140,104,1,0,W 1,19480,2:17,NaN
1,2,Fri Oct 25 2024,9:30p,NaN,Box Score,@,Utah Jazz,W,NaN,127,86,2,0,W 2,18175,2:07,NaN
2,3,Sun Oct 27 2024,8:30p,NaN,Box Score,NaN,Los Angeles Clippers,L,NaN,104,112,2,1,L 1,18064,2:22,NaN
3,4,Tue Oct 29 2024,10:00p,NaN,Box Score,NaN,New Orleans Pelicans,W,NaN,124,106,3,1,W 1,18064,2:20,NaN
4,5,Wed Oct 30 2024,10:00p,NaN,Box Score,NaN,New Orleans Pelicans,W,NaN,104,89,4,1,W 2,18064,2:19,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,79,Tue Apr 8 2025,10:00p,NaN,Box Score,@,Phoenix Suns,W,NaN,133,95,47,32,W 1,17071,2:09,NaN
79,80,Wed Apr 9 2025,10:00p,NaN,Box Score,NaN,San Antonio Spurs,L,NaN,111,114,47,33,L 1,18064,2:17,NaN
80,81,Fri Apr 11 2025,10:00p,NaN,Box Score,@,Portland Trail Blazers,W,NaN,103,86,48,33,W 1,19335,2:08,NaN
81,82,Sun Apr 13 2025,3:30p,NaN,Box Score,NaN,Los Angeles Clippers,L,OT,119,124,48,34,L 1,18064,2:42,NaN


In [16]:
df = df[['G', 'Tm']]
df = df.rename(columns={"Tm": "ppg"})
df

,G,ppg
0,1,140
1,2,127
2,3,104
3,4,124
4,5,104
...,...,...
78,79,133
79,80,111
80,81,103
81,82,119


**Visualizando...**

In [17]:
fig = px.line(df, x='G', y='ppg', title='Puntos Por Juego por los Warriors, 2024-2025')
fig.show()

No teníamos datos faltantes o incorrectos así que no se hizo algún proceso de limpieza de datos. Con los datos cargados, podemos dividir el periodo de entrenamiento y prueba.

Se va a utilizar una ventana deslizante de 10 juegos, ya que esta es aproximadamente la cantidad de partidos que se jugarían en un mes durante la temporada regular de la NBA. Vamos a revisar 7 juegos al final de la temporada como nuestros valores de prueba.

In [ ]:
train_size = int((len(df)* 0.8))
train_data = df[:train_size]
test_data = df[train_size:]

scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train_data[['ppg']])
test_scaled = scaler.transform(test_data[['ppg']])

# Función para crear ventanas deslizantestrain_size = int((len(df)* 0.8))
train_data = df[:train_size]
test_data = df[train_size:]

scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train_data[['ppg']])
test_scaled = scaler.transform(test_data[['ppg']])

# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 10

X_train, y_train = crear_ventanas(train_scaled, WINDOW_SIZE)
X_test, y_test = crear_ventanas(test_scaled, WINDOW_SIZE)
train_size = int((len(df)* 0.8))
train_data = df[:train_size]
test_data = df[train_size:]

scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train_data[['ppg']])
test_scaled = scaler.transform(test_data[['ppg']])

# Función para crear ventanas deslizantestrain_size = int((len(df)* 0.8))
train_data = df[:train_size]
test_data = df[train_size:]

scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train_data[['ppg']])
test_scaled = scaler.transform(test_data[['ppg']])

# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 10

X_train, y_train = crear_ventanas(train_scaled, WINDOW_SIZE)
X_test, y_test = crear_ventanas(test_scaled, WINDOW_SIZE)

X_train = X_train.reshape(X_train.shape[0], WINDOW_SIZE)
X_test = X_test.reshape(X_test.shape[0], WINDOW_SIZE)
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 10

X_train, y_train = crear_ventanas(train_scaled, WINDOW_SIZE)
X_test, y_test = crear_ventanas(test_scaled, WINDOW_SIZE)

X_train = X_train.reshape(X_train.shape[0], WINDOW_SIZE)
X_test = X_test.reshape(X_test.shape[0], WINDOW_SIZE)
X_train = X_train.reshape(X_train.shape[0], WINDOW_SIZE)
X_test = X_test.reshape(X_test.shape[0], WINDOW_SIZE)
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

WINDOW_SIZE = 10

X_train, y_train = crear_ventanas(train_scaled, WINDOW_SIZE)
X_test, y_test = crear_ventanas(test_scaled, WINDOW_SIZE)

X_train = X_train.reshape(X_train.shape[0], WINDOW_SIZE)
X_test = X_test.reshape(X_test.shape[0], WINDOW_SIZE)

In [19]:
last_window = test_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE)
future_scaled = []

La red neuronal tiene una forma muy sencilla, con tan solo tres capas Densas de 64, 32 y 1 neurona respectivamente. Hicimos varias pruebas con distintos números de capas y neuronas pero este fue el que mejor funcionó en prueba y error. Al ser una red tan sencilla, definitivamente se podría expandir a futuro con una búsqueda tipo GridSearch con otras arquitecturas, funciones de activación, optimizadores, etc.

In [28]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(WINDOW_SIZE,)),
    Dense(32, activation='relu'),
    Dense(1)
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=Adam(), loss='mse')
model.fit(X_train, y_train,
          epochs=100, validation_data=(X_test, y_test))

In [25]:
y_pred = model.predict(X_test)

y_pred_real = scaler.inverse_transform(y_pred)
y_test_real = scaler.inverse_transform(y_test.reshape(-1,1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step


**Visualizando resultados...**

In [32]:
n = 7

test_x = np.arange(len(train_data), len(train_data) + 7)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test_x,
    y=y_test_real.flatten()[-n:],
    mode='lines+markers',
    name='Real',
    marker=dict(symbol='circle')
))

fig.add_trace(go.Scatter(
    x=test_x,
    y=y_pred_real.flatten()[-n:],
    mode='lines+markers',
    name='Predicted',
    marker=dict(symbol='x')
))

fig.update_layout(
    title="Ultimos 7 Juegos",
    xaxis_title="Juego",
    yaxis_title="Puntos",
)

fig.show()

In [27]:
mae = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print(f'MAE: {mae}')
print(f'RMSE: {rmse}')

MAE: 11.059526715959821
RMSE: 14.155870861733993


### <font color= #99C8F5> **Conclusiones** </font>

El modelo es bastante decente en sus predicciones. Su error promedio es de unos 11 puntos, el cuál es un resultado muy decente considerando la volatilidad que existe en la NBA para el rendimiento de un equipo debido a las heridas que pueden afectar a jugadores estrella o las suspensiones de los mismos.

Viéndo a futuro, podríamos tomar en cuenta otras variables al predecir los puntos. La más obvia es el equipo contra el que se está jugando, ya que es evidente que jugar contra un equipo decente y contra uno malo va a dar resultados distintos. También podríamos considerar heridas de jugadores, si están jugando en casa o de visitantes, etcétera. No tomaríamos datos de años anteriores ya que los equipos pueden cambiar mucho de temporada a temporada y no siempre presentan consistencia, sobre todo en ligas con límites de salario como en la NBA. 
